In [1]:
import torch
from torch import nn

In [2]:
class customLayerNorm(nn.Module):
    def __init__(self, normalizer_shape, eps=1e-5, elementwise_affine=True):
        super().__init__()
        if type(normalizer_shape)==int:
            normalizer_shape = (normalizer_shape,)
        self.normalizer_shape = normalizer_shape
        
        self.eps = eps
        self.elementwise_affine = elementwise_affine
        if elementwise_affine:
            self.gamma = nn.Parameter(torch.ones(*self.normalizer_shape))
            self.beta = nn.Parameter(torch.zeros(*self.normalizer_shape))
        else:
            self.register_parameter('gamma', None)
            self.register_parameter('beta', None)
    
    def forward(self, x, dim = -1):
        
        mean = torch.mean(x, keepdim=True,dim=dim)
        # 有偏的方差 无偏方差
        var = torch.var(x, keepdim=True,dim=dim, unbiased=False)
        
        #broadcast机制 pytorch，numpy
        if self.elementwise_affine:
            output = self.gamma*(x-mean)/(torch.sqrt(var)+self.eps) +self.beta
        else:
            output = (x-mean)/(torch.sqrt(var)+self.eps)
        
        return output
        

In [3]:
batch_size, seq_length, hidden_size = 16, 20, 768
x = torch.randn(batch_size, seq_length, hidden_size)

ln = customLayerNorm(hidden_size)

print("输入形状:", x.shape)
print("输入均值:", x.mean(dim=-1))
print("输入方差:", x.var(dim=-1, unbiased=False))

result = ln(x)
print("输出形状:", result.shape)
print("输出均值:", result.mean(dim=-1))
print("输出方差:", result.var(dim=-1, unbiased=False))


输入形状: torch.Size([16, 20, 768])
输入均值: tensor([[ 0.0459,  0.0413,  0.0236, -0.0701, -0.0069,  0.0290,  0.0027, -0.0258,
         -0.0134,  0.0002,  0.0104,  0.0246,  0.0071,  0.0105, -0.0501, -0.0507,
          0.0124, -0.0128, -0.0469, -0.0539],
        [ 0.0228,  0.0129, -0.0136,  0.0308, -0.0196, -0.0429, -0.0150, -0.0400,
         -0.0088, -0.0580,  0.0136,  0.0593,  0.0377,  0.0386, -0.0092, -0.0210,
          0.0204, -0.0367, -0.0216, -0.0027],
        [ 0.0180, -0.0149, -0.0171,  0.0026,  0.0673, -0.0488, -0.0030, -0.0569,
         -0.0570, -0.0409,  0.0243,  0.0024,  0.0353,  0.0585, -0.0004,  0.0502,
         -0.0575, -0.0075,  0.0600, -0.0376],
        [ 0.0217, -0.0738,  0.0422,  0.0074, -0.0325, -0.0474,  0.0089,  0.0596,
         -0.0427,  0.0013, -0.0430, -0.0039, -0.0084, -0.0604,  0.0639, -0.0492,
          0.0262, -0.0020,  0.0482,  0.0078],
        [ 0.0127, -0.0659, -0.0284, -0.0304, -0.0547,  0.0079,  0.0288, -0.0469,
          0.0259, -0.0298, -0.0791, -0.0126, -0.0

In [6]:
# softmax实现

def softmax_torch(x, dim=-1):
    # e
    max_values = torch.max(x, dim=dim, keepdim=True).values
    e_x = torch.exp(x-max_values)
    output = e_x/torch.sum(e_x, dim = dim, keepdim=True)
    return output 

In [10]:
batch_size, seq_len = 16, 10
x = torch.randn(batch_size, seq_len)

print(f"x is {x}")
result = softmax_torch(x, dim=-1)
print(f"result is {result}")
print(f"sum of result is {torch.sum(result, -1)}")

x is tensor([ 0.2849, -0.5605, -0.0709,  1.4629,  0.3776,  1.4905,  0.2230, -0.7794,
         0.5499, -1.3875])
result is tensor([0.0794, 0.0341, 0.0557, 0.2580, 0.0871, 0.2652, 0.0747, 0.0274, 0.1035,
        0.0149])
sum of result is 1.0


In [12]:
def softmax_temparature(x, temparature = 1.0, dim=-1):
    max_values = torch.max(x, dim=dim, keepdim=True).values
    e_x = torch.exp((x-max_values)/temparature)
    output = e_x/torch.sum(e_x, dim = dim, keepdim=True)
    return output

In [13]:
batch_size, seq_len = 16, 10
x = torch.randn(seq_len)

print(f"x is {x}")
result = softmax_temparature(x, temparature=1.0, dim=-1)
print(f"result is {result}")

x is tensor([-0.7271, -1.4125,  0.9952,  0.2788,  0.7083,  0.0493,  0.2220,  0.3912,
         1.1176,  0.3263])
result is tensor([0.0322, 0.0162, 0.1803, 0.0881, 0.1353, 0.0700, 0.0832, 0.0985, 0.2038,
        0.0924])


In [14]:
print(f"x is {x}")
result = softmax_temparature(x, temparature=0.2, dim=-1)
print(f"result is {result}")

x is tensor([-0.7271, -1.4125,  0.9952,  0.2788,  0.7083,  0.0493,  0.2220,  0.3912,
         1.1176,  0.3263])
result is tensor([5.6454e-05, 1.8338e-06, 3.1020e-01, 8.6259e-03, 7.3889e-02, 2.7388e-03,
        6.4952e-03, 1.5135e-02, 5.7192e-01, 1.0943e-02])


In [15]:
print(f"x is {x}")
result = softmax_temparature(x, temparature=1.2, dim=-1)
print(f"result is {result}")

x is tensor([-0.7271, -1.4125,  0.9952,  0.2788,  0.7083,  0.0493,  0.2220,  0.3912,
         1.1176,  0.3263])
result is tensor([0.0398, 0.0225, 0.1674, 0.0921, 0.1318, 0.0761, 0.0879, 0.1012, 0.1853,
        0.0959])


In [49]:
import numpy as np
1/(1+np.exp(-2*0.5 ))

0.7310585786300049